**Imports**

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split, Subset

**Configuration**

In [ ]:
TRAIN_CSV = "../data/kaggle/fashion-mnist_train.csv"
TEST_CSV = "../data/kaggle/fashion-mnist_test.csv"

BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
K_FOLDS = 5
DROPOUT_RATE = 0.5
RANDOM_SEED = 42

CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


**Check files**

In [3]:
if not os.path.exists(TRAIN_CSV):
    raise FileNotFoundError(f"Training file not found: {TRAIN_CSV}")

if not os.path.exists(TEST_CSV):
    raise FileNotFoundError(f"Test file not found: {TEST_CSV}")

print("Both files found.")

Both files found.


**Create custom dataset**

In [4]:
class FashionMNISTCSVDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)

        self.labels = df.iloc[:, 0].to_numpy(dtype=np.int64)
        self.images = df.iloc[:, 1:].to_numpy(dtype=np.float32) / 255.0  # / 255 part nomalizes pixel values, become [0,1]
        self.images = self.images.reshape(-1, 1, 28, 28) # (number_of_images, channels, height, width)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx): # get the sample at position idx
        image = torch.from_numpy(self.images[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

**Load datasets**

In [5]:
full_train_dataset = FashionMNISTCSVDataset(TRAIN_CSV)
test_dataset = FashionMNISTCSVDataset(TEST_CSV)

print("Full train dataset size:", len(full_train_dataset))
print("Test dataset size:", len(test_dataset))

Full train dataset size: 60000
Test dataset size: 10000


**Prepare manual K-fold splits**

In [ ]:
indices = np.arange(len(full_train_dataset))
np.random.seed(RANDOM_SEED)
np.random.shuffle(indices)

fold_indices = np.array_split(indices, K_FOLDS) # split the indices into K_FOLDS equal parts 

for i, fold_idx in enumerate(fold_indices, start=1):
    print(f"Fold {i} size: {len(fold_idx)}")


Fold 1 size: 12000
Fold 2 size: 12000
Fold 3 size: 12000
Fold 4 size: 12000
Fold 5 size: 12000


**Create test DataLoader**

In [7]:
pin_memory = torch.cuda.is_available()
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

**Define the CNN model**

In [8]:
class FashionCNNFlexible(nn.Module):
    def __init__(
        self,
        conv_channels=[32, 64],
        kernel_size=3,
        pool_type="max",
        activation="relu",
        dropout=0.3,
        use_batchnorm=False,
        num_classes=10
    ):
        super().__init__()

        layers = []
        in_channels = 1
        current_size = 28  # Fashion-MNIST image size

        # choose activation
        def get_activation():
            if activation.lower() == "relu":
                return nn.ReLU()
            elif activation.lower() == "gelu":
                return nn.GELU()
            else:
                raise ValueError(f"Unsupported activation: {activation}")

        # choose pooling
        def get_pool():
            if pool_type.lower() == "max":
                return nn.MaxPool2d(kernel_size=2, stride=2)
            elif pool_type.lower() == "avg":
                return nn.AvgPool2d(kernel_size=2, stride=2)
            else:
                raise ValueError(f"Unsupported pool type: {pool_type}")

        padding = kernel_size // 2  # keep spatial size same after conv

        for out_channels in conv_channels:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding))
            
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))
            
            layers.append(get_activation())
            layers.append(get_pool())

            in_channels = out_channels
            current_size = current_size // 2  # after 2x2 pooling

        self.features = nn.Sequential(*layers)

        flattened_dim = conv_channels[-1] * current_size * current_size

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_dim, 128),
            get_activation(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [9]:
model_configs = {
    "V1_Baseline": {
        "conv_channels": [32, 64],
        "kernel_size": 3,
        "pool_type": "max",
        "activation": "relu",
        "dropout": 0.3,
        "use_batchnorm": False
    },
    "V2_BatchNorm": {
        "conv_channels": [32, 64],
        "kernel_size": 3,
        "pool_type": "max",
        "activation": "relu",
        "dropout": 0.3,
        "use_batchnorm": True
    },
    "V3_LargerKernel": {
        "conv_channels": [32, 64],
        "kernel_size": 5,
        "pool_type": "max",
        "activation": "relu",
        "dropout": 0.3,
        "use_batchnorm": False
    },
    "V4_AvgPool": {
        "conv_channels": [32, 64],
        "kernel_size": 3,
        "pool_type": "avg",
        "activation": "relu",
        "dropout": 0.3,
        "use_batchnorm": False
    },
    "V5_GELU": {
        "conv_channels": [32, 64],
        "kernel_size": 3,
        "pool_type": "max",
        "activation": "gelu",
        "dropout": 0.3,
        "use_batchnorm": False
    },
    "V6_NoDropout": {
        "conv_channels": [32, 64],
        "kernel_size": 3,
        "pool_type": "max",
        "activation": "relu",
        "dropout": 0.0,
        "use_batchnorm": False
    },
    "V7_Deeper": {
        "conv_channels": [16, 32, 64],
        "kernel_size": 3,
        "pool_type": "max",
        "activation": "relu",
        "dropout": 0.3,
        "use_batchnorm": True
    }
}

**Initialize model, loss, optimizer**

In [10]:
model = FashionCNNFlexible().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

FashionCNNFlexible(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


**Training function**

In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader: # images: [batch_size, 1, 28, 28], labels: [batch_size]
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images) # [batch_size, 10]
        loss = criterion(outputs, labels) # computes loss 
        loss.backward() # calculates graadients
        optimizer.step() # updates the model parameters based on the calculated gradients

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1) # predicted class is the index of the largest value, [batch_size]
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

**Evaluation function**

In [12]:
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): # tells PyTorch not to calculate gradients, which saves memory and computations during evaluation
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy()) # This saves all predictions and actual labels across all batches
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
all_cv_results = []

os.makedirs("transformers/saved_models_kfold", exist_ok=True)
os.makedirs("transformers/results_kfold", exist_ok=True)

for model_name, config in model_configs.items():
    print(f"\nCross-validating {model_name}...")

    fold_results = []

    for fold in range(K_FOLDS):
        print(f"\n--- Fold {fold + 1}/{K_FOLDS} for {model_name} ---")

        val_idx = fold_indices[fold]
        train_idx = np.concatenate([fold_indices[i] for i in range(K_FOLDS) if i != fold])

        train_subset = Subset(full_train_dataset, train_idx)
        val_subset = Subset(full_train_dataset, val_idx)

        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

        model = FashionCNNFlexible(**config).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        train_losses = []
        train_accuracies = []
        val_losses = []
        val_accuracies = []

        best_val_acc = 0.0
        best_val_acc_epoch = 0

        best_val_loss = float("inf")
        best_val_loss_epoch = 0

        for epoch in range(NUM_EPOCHS):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

            train_losses.append(train_loss)
            train_accuracies.append(train_acc)
            val_losses.append(val_loss)
            val_accuracies.append(val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_val_acc_epoch = epoch + 1
                torch.save(
                    model.state_dict(),
                    f"saved_models_kfold/{model_name}_fold{fold + 1}_best_acc.pth"
                )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_val_loss_epoch = epoch + 1
                torch.save(
                    model.state_dict(),
                    f"saved_models_kfold/{model_name}_fold{fold + 1}_best_loss.pth"
                )

            print(
                f"{model_name} | Fold [{fold + 1}/{K_FOLDS}] | Epoch [{epoch + 1}/{NUM_EPOCHS}]"
            )
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
            print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
            print("-" * 50)

        fold_log_df = pd.DataFrame({
            "model": model_name,
            "fold": fold + 1,
            "epoch": range(1, NUM_EPOCHS + 1),
            "train_loss": train_losses,
            "train_accuracy": train_accuracies,
            "val_loss": val_losses,
            "val_accuracy": val_accuracies
        })
        fold_log_df.to_csv(f"results_kfold/{model_name}_fold{fold + 1}_training_log.csv", index=False)

        fold_summary = {
            "model": model_name,
            "fold": fold + 1,
            "best_val_accuracy": best_val_acc,
            "best_val_accuracy_epoch": best_val_acc_epoch,
            "best_val_loss": best_val_loss,
            "best_val_loss_epoch": best_val_loss_epoch,
            "final_train_loss": train_losses[-1],
            "final_train_accuracy": train_accuracies[-1],
            "final_val_loss": val_losses[-1],
            "final_val_accuracy": val_accuracies[-1]
        }

        fold_results.append(fold_summary)
        all_cv_results.append(fold_summary)

    model_fold_df = pd.DataFrame(fold_results)

    model_cv_summary = {
        "model": model_name,
        "mean_best_val_accuracy": model_fold_df["best_val_accuracy"].mean(),
        "std_best_val_accuracy": model_fold_df["best_val_accuracy"].std(),
        "mean_best_val_loss": model_fold_df["best_val_loss"].mean(),
        "std_best_val_loss": model_fold_df["best_val_loss"].std(),
        "mean_final_val_accuracy": model_fold_df["final_val_accuracy"].mean(),
        "std_final_val_accuracy": model_fold_df["final_val_accuracy"].std(),
        "mean_final_val_loss": model_fold_df["final_val_loss"].mean(),
        "std_final_val_loss": model_fold_df["final_val_loss"].std()
    }

    pd.DataFrame([model_cv_summary]).to_csv(
        f"results_kfold/{model_name}_cv_summary.csv", index=False
    )

all_cv_df = pd.DataFrame(all_cv_results)
all_cv_df.to_csv("results_kfold/all_models_all_folds.csv", index=False)

cv_summary_df = (
    all_cv_df.groupby("model", as_index=False)
    .agg(
        mean_best_val_accuracy=("best_val_accuracy", "mean"),
        std_best_val_accuracy=("best_val_accuracy", "std"),
        mean_best_val_loss=("best_val_loss", "mean"),
        std_best_val_loss=("best_val_loss", "std"),
        mean_final_val_accuracy=("final_val_accuracy", "mean"),
        std_final_val_accuracy=("final_val_accuracy", "std"),
        mean_final_val_loss=("final_val_loss", "mean"),
        std_final_val_loss=("final_val_loss", "std")
    )
    .sort_values(by="mean_best_val_accuracy", ascending=False)
)

cv_summary_df.to_csv("results_kfold/model_comparison_cv_summary.csv", index=False)

cv_summary_by_loss_df = cv_summary_df.sort_values(by="mean_best_val_loss", ascending=True)
cv_summary_by_loss_df.to_csv("results_kfold/model_comparison_cv_summary_by_loss.csv", index=False)

print("\nCross-validation completed.")
print("\nSorted by mean best validation accuracy:")
print(cv_summary_df)

print("\nSorted by mean best validation loss:")
print(cv_summary_by_loss_df)



Cross-validating V1_Baseline...

--- Fold 1/5 for V1_Baseline ---
V1_Baseline | Fold [1/5] | Epoch [1/50]
Train Loss: 0.5610 | Train Acc: 0.7946
Val   Loss: 0.3529 | Val   Acc: 0.8705
--------------------------------------------------
V1_Baseline | Fold [1/5] | Epoch [2/50]
Train Loss: 0.3604 | Train Acc: 0.8693
Val   Loss: 0.3098 | Val   Acc: 0.8848
--------------------------------------------------
V1_Baseline | Fold [1/5] | Epoch [3/50]
Train Loss: 0.3094 | Train Acc: 0.8889
Val   Loss: 0.2725 | Val   Acc: 0.8992
--------------------------------------------------
V1_Baseline | Fold [1/5] | Epoch [4/50]
Train Loss: 0.2747 | Train Acc: 0.9005
Val   Loss: 0.2565 | Val   Acc: 0.9048
--------------------------------------------------
V1_Baseline | Fold [1/5] | Epoch [5/50]
Train Loss: 0.2477 | Train Acc: 0.9119
Val   Loss: 0.2434 | Val   Acc: 0.9116
--------------------------------------------------
V1_Baseline | Fold [1/5] | Epoch [6/50]
Train Loss: 0.2247 | Train Acc: 0.9168
Val   Los